In [1]:
import mysql.connector

# configuración de conexión a la base de datos
config: dict = {
    'user': 'root',
    'password': 'kP92mXv7R4wQ',
    'host': '127.0.0.1',
    'database': 'asteroides_listo_para_sql'
}

def visualizar_impacto_agrupado() -> None:
    """genera una tabla donde cada asteroide muestra todas las localizaciones."""
    try:
        conn = mysql.connector.connect(**config)
        cursor = conn.cursor()

        # usamos cross join para obtener todas las combinaciones (25 filas)
        # ordenamos primero por dimensión y luego por nombre para la agrupación
        query: str = """
        with ast_top as (
            select a.name, a.velocity_km_h, d.diameter_avg_m
            from asteroides_ranking a
            left join dimension_ranking d on a.name = d.name 
            limit 5
        ),
        grav_top as (
            select localization, gravity_value
            from gravity.gravity 
            limit 5
        )
        select 
            a.name,
            a.diameter_avg_m,
            g.localization,
            g.gravity_value,
            a.velocity_km_h
        from ast_top a
        cross join grav_top g
        order by a.diameter_avg_m desc, a.name asc;
        """

        cursor.execute(query)
        dataset: list[tuple] = cursor.fetchall()

        # impresión con formato de agrupación
        print(f"{'Entidad':<25} | {'Detalle Físico':<20} | {'Gravedad/Velocidad':<20}")
        print("=" * 70)

        asteroide_actual: str = ""

        for fila in dataset:
            nombre, dim, loc, grav, vel = fila
            
            # si el nombre cambia, imprimimos una cabecera para el nuevo asteroide
            if nombre != asteroide_actual:
                asteroide_actual = nombre
                print(f"\nASTEROIDE: {nombre.upper()}")
                print(f"Dimensión: {dim:,.2f} m | Velocidad: {vel:,.2f} km/h")
                print("-" * 70)
            
            # imprimimos las localizaciones asociadas a ese asteroide
            print(f"  -> {loc:<22} | Gravedad: {grav:>10.4f} mGal")

    except Exception as e:
        print(f"error al procesar la tabla: {e}")
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()

if __name__ == "__main__":
    visualizar_impacto_agrupado()

Entidad                   | Detalle Físico       | Gravedad/Velocidad  

ASTEROIDE: ABHRAMU
Dimensión: 2,020.89 m | Velocidad: 14,773.70 km/h
----------------------------------------------------------------------
  -> Monte Everest          | Gravedad:   663.1850 mGal
  -> Fosa de las Marianas   | Gravedad:  -318.5890 mGal
  -> Ciudad de México       | Gravedad:    33.5749 mGal
  -> Castellón de la Plana  | Gravedad:     5.4615 mGal
  -> Cuenca de Tanezrouft   | Gravedad:   -11.4702 mGal

ASTEROIDE: AHAU
Dimensión: 1,443.92 m | Velocidad: 41,062.60 km/h
----------------------------------------------------------------------
  -> Monte Everest          | Gravedad:   663.1850 mGal
  -> Fosa de las Marianas   | Gravedad:  -318.5890 mGal
  -> Ciudad de México       | Gravedad:    33.5749 mGal
  -> Castellón de la Plana  | Gravedad:     5.4615 mGal
  -> Cuenca de Tanezrouft   | Gravedad:   -11.4702 mGal

ASTEROIDE: ADONIS
Dimensión: 786.22 m | Velocidad: 69,587.40 km/h
----------------------